# 05 — Full-resolution validation of saved checkpoints (Phase 2)

**No training.** Loads `ckpt_best.pth` of runs made with notebook 04 (EXP-001, EXP-002, ...) and evaluates them on **every original point**
of the validation stations S163 + S169, not only on the 2 cm sub-sampled points used during training.

Why: at 2 cm the validation stations hold only 27 switch, 34 outlet and 46 damper points, and both baselines score IoU 0.00 on them.
At full resolution those classes have thousands of points.

What it reports, per checkpoint:
1. **2cm** — the same evaluation as during training (must reproduce the number stored in the checkpoint);
2. **full<-2cm** — the 2 cm predictions copied to all original points (each point takes the label of its voxel's representative);
3. **full<-1cm** — exploratory: the same network applied to 1 cm points, copied to all original points;
4. for Components and the five small-component classes: **which classes their points are predicted as**.

Model-free part (runs on CPU too): the **oracle** — the score obtained if every voxel representative were classified perfectly.
It is the ceiling that the voxel size alone puts on each class.

**How to run:** Add Input → Your Work → Notebooks → `04_baseline_pointnet2`, Settings → Accelerator **GPU T4 x2**,
Save Version → *Save & Run All* (Advanced: "Run with GPU for this session"). About 3 minutes per checkpoint.

**Which checkpoint?** In a saved run Kaggle attaches the **newest** version of notebook 04's output; the editor has no version picker.
So: evaluate a run while it is the newest version. For an older version (version 2 = EXP-001, version 3 = EXP-002) work in an **interactive session**:
set `nb04_version` in the first code cell (or run `import kagglehub; kagglehub.notebook_output_download("<your-kaggle-user>/04-baseline-pointnet2/versions/2")`
in the editor console `>_`), press **Run All** with the GPU on, read the results in the editor and stop the session.
Save Version does not keep the older attachment — a saved run then fails with "New Notebooks cannot be attached in non-interactive sessions"
(this happened in versions 3 and 4 on Kaggle; EVAL-001 used version 2 for EXP-002 and an interactive session for EXP-001).

On CPU (the automatic run after import) it computes the data tables and the oracle (real numbers) and tests the prediction code with a stand-in model (meaningless numbers).

In [ ]:
import os, sys, json, time, glob, math, subprocess
import numpy as np

CFG = dict(
    eval_name="EVAL-001_fullres_val",
    val=["S163", "S169"],
    small=[3, 5, 6, 10, 14],          # Damper, Exit sign, Fire terminal, Outlet, Switch
    focus=[1, 3, 5, 6, 10, 14],       # classes whose confusions are printed: Components + the five small ones
    num_classes=18,
    also_1cm=True,                    # also apply the network to 1 cm points (exploratory)
    group=4,                          # crops per forward pass (as in notebook 04)
    nb04="<your-kaggle-user>/04-baseline-pointnet2",   # your copy of notebook 04 on Kaggle
    nb04_version=None,                # e.g. 2 -> attach that version of notebook 04's output; INTERACTIVE SESSIONS ONLY (see "Which checkpoint?")
)
CLASS_NAMES = ["Column", "Components", "Covering", "Damper", "Door", "Exit sign", "Fire terminal", "Furniture",
               "Heater", "Lamp", "Outlet", "Railing", "Slab", "Stair", "Switch", "Wall", "Window", "Clutter"]
VOXELS = {"2cm": 0.02, "1cm": 0.01}
OFFSET = np.array([900.0, 480.0, 100.0])
C = CFG["num_classes"]
ON_KAGGLE = os.path.isdir("/kaggle/working")
OUT = os.path.join("/kaggle/working" if ON_KAGGLE else "./results", CFG["eval_name"])
os.makedirs(OUT, exist_ok=True)
print(json.dumps(CFG, indent=1)); print("OUT:", OUT)

## Data: original points of the validation stations, their 2 cm and 1 cm sub-samples, and the map between them

The sub-sampling is the same procedure as in notebooks 03 and 04 (one point per voxel: the one closest to the voxel centre).
`inv[i]` is the position, in the sub-sampled cloud, of the representative of original point `i`.

In [ ]:
def voxel_subsample_inv(xyz, size):
    """Same selection as voxel_subsample() of notebook 04, plus the map original point -> its voxel's representative."""
    g = np.floor(xyz / size).astype(np.int64)
    assert g.min() >= 0 and g.max() < (1 << 20)
    key = g[:, 0] | (g[:, 1] << 20) | (g[:, 2] << 40)
    d2 = ((xyz - (g + 0.5) * size) ** 2).sum(1)
    order = np.lexsort((d2, key)); ks = key[order]
    first = np.ones(len(order), dtype=bool); first[1:] = ks[1:] != ks[:-1]
    inv = np.empty(len(order), dtype=np.int64); inv[order] = np.cumsum(first) - 1
    return order[first], inv

def get_raw_dir():
    need = lambda root: all(os.path.isfile(os.path.join(root, "Gold", s + ".npy")) for s in CFG["val"])
    for root in [os.environ.get("RAW_DIR", ""), "/kaggle/temp/3dses", "./data/3dses"]:
        if root and need(root): return root
    try:
        from huggingface_hub import snapshot_download
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import snapshot_download
    raw = "/kaggle/temp/3dses" if ON_KAGGLE else "./data/3dses"
    snapshot_download(repo_id="heig-vd-geo/3DSES", repo_type="dataset", allow_patterns=["Gold/" + s + ".npy" for s in CFG["val"]], local_dir=raw)
    assert need(raw), "download did not produce the expected files"
    return raw

from scipy.spatial import cKDTree
RAW = get_raw_dir(); print("raw data:", RAW)
FULL, SUB = {}, {v: {} for v in VOXELS}
for s in CFG["val"]:
    a = np.load(os.path.join(RAW, "Gold", s + ".npy"))
    assert a.ndim == 2 and a.shape[1] >= 8, a.shape
    xyz = a[:, :3] - OFFSET
    assert xyz.min() >= 0 and xyz.max() < 250, "unexpected coordinates - check OFFSET"
    lab = a[:, 7].astype(np.int64); assert lab.min() >= 0 and lab.max() < C
    FULL[s] = dict(label=lab, n=len(lab))
    for v, size in VOXELS.items():
        keep, inv = voxel_subsample_inv(xyz, size)
        assert (np.floor(xyz[keep][inv] / size) == np.floor(xyz / size)).all()      # every point maps to a representative in its own voxel
        st = dict(xyz=xyz[keep].astype(np.float32), rgb=np.clip(a[keep, 3:6], 0, 255).astype(np.uint8),
                  intensity=a[keep, 6].astype(np.float32), label=lab[keep], inv=inv)
        st["tree"] = cKDTree(st["xyz"]); SUB[v][s] = st
    print(f"{s}: {len(lab):>10,} original points | " + " | ".join(f"{v}: {len(SUB[v][s]['xyz']):>9,}" for v in VOXELS))
    del a, xyz

def count(labels): return np.bincount(labels, minlength=C)
N_FULL = sum(count(FULL[s]["label"]) for s in CFG["val"])
N_SUB = {v: sum(count(SUB[v][s]["label"]) for s in CFG["val"]) for v in VOXELS}
print(f"\nvalidation points per class ({' + '.join(CFG['val'])})")
print(f"{'ID':>2} {'class':<14} {'original':>11} " + " ".join(f"{v:>9}" for v in VOXELS))
for c in range(C):
    if N_FULL[c]:
        print(f"{c:>2} {CLASS_NAMES[c]:<14} {N_FULL[c]:>11,} " + " ".join(f"{N_SUB[v][c]:>9,}" for v in VOXELS) + ("   <- small component" if c in CFG["small"] else ""))
print(f"{'':>2} {'all':<14} {N_FULL.sum():>11,} " + " ".join(f"{N_SUB[v].sum():>9,}" for v in VOXELS))

## Oracle: the ceiling set by the voxel size (no model involved)

If the network labelled every voxel representative correctly and that label were copied to all original points of the voxel,
this is the score it would get on the original points. Anything below 100 is lost before the network sees the data.

In [ ]:
def confusion(gt, pred, C):
    return np.bincount(gt * C + pred, minlength=C * C).reshape(C, C)

def metrics_from_confusion(cm, small):
    tp = np.diag(cm).astype(np.float64); gt = cm.sum(1); pr = cm.sum(0)
    present = gt > 0
    iou = np.where(present, tp / np.maximum(gt + pr - tp, 1), np.nan)
    acc = np.where(present, tp / np.maximum(gt, 1), np.nan)
    sm = [c for c in small if present[c]]
    return dict(mIoU=float(np.nanmean(iou) * 100), mAcc=float(np.nanmean(acc) * 100), OA=float(tp.sum() / max(cm.sum(), 1) * 100),
                small_mIoU=float(np.mean([iou[c] for c in sm]) * 100) if sm else float("nan"),
                iou=[None if np.isnan(v) else round(float(v) * 100, 2) for v in iou], n_classes_present=int(present.sum()))

def headline(rows):
    """rows: list of (name, metrics dict)"""
    w = max(len(n) for n, _ in rows) + 1
    print(f"{'':<{w}} {'mIoU':>7} {'small mIoU':>11} {'mAcc':>7} {'OA':>7}")
    for n, m in rows: print(f"{n:<{w}} {m['mIoU']:>7.2f} {m['small_mIoU']:>11.2f} {m['mAcc']:>7.2f} {m['OA']:>7.2f}")

def per_class(cols, counts=None):
    """cols: list of (name, metrics dict); prints IoU per class, one column per entry"""
    print(f"{'ID':>2} {'class':<14} " + (f"{'points':>11} " if counts is not None else "") + " ".join(f"{n:>10}" for n, _ in cols))
    for c in range(C):
        if any(m["iou"][c] is not None for _, m in cols):
            print(f"{c:>2} {CLASS_NAMES[c]:<14} " + (f"{counts[c]:>11,} " if counts is not None else "") +
                  " ".join(f"{m['iou'][c]:>10.2f}" if m["iou"][c] is not None else f"{'-':>10}" for _, m in cols) +
                  ("   <- small component" if c in CFG["small"] else ""))

ORACLE = {}
for v in VOXELS:
    cm = sum(confusion(FULL[s]["label"], SUB[v][s]["label"][SUB[v][s]["inv"]], C) for s in CFG["val"])
    assert cm.sum() == N_FULL.sum()
    ORACLE[v] = metrics_from_confusion(cm, CFG["small"])
print("ORACLE on the original validation points (perfect labels on the voxel representatives)\n")
headline([("oracle " + v, ORACLE[v]) for v in VOXELS])
print("\nIoU % per class"); per_class([("oracle " + v, ORACLE[v]) for v in VOXELS], N_FULL)
with open(os.path.join(OUT, "oracle.json"), "w") as fh:
    json.dump({"val": CFG["val"], "points_original": N_FULL.tolist(), "points_sub": {v: N_SUB[v].tolist() for v in VOXELS},
               "oracle": ORACLE, "class_names": CLASS_NAMES}, fh, indent=1)

## Prediction code (NumPy only; identical crops and features to notebook 04's validation)

In [ ]:
def crop_indices(st, center_idx, k):
    _, idx = st["tree"].query(st["xyz"][center_idx], k=min(k, len(st["xyz"])))
    return np.asarray(idx)

def make_sample(st, idx, use_intensity):
    """Evaluation branch of make_sample() in notebook 04: pos [n,3], feat [C,n]."""
    xyz = st["xyz"][idx].astype(np.float32).copy()
    rgb = st["rgb"][idx].astype(np.float32) / 127.5 - 1.0
    pos = xyz - np.array([xyz[:, 0].mean(), xyz[:, 1].mean(), xyz[:, 2].min()], dtype=np.float32)
    feats = [rgb, pos[:, 2:3]]
    if use_intensity: feats.append((st["intensity"][idx] * 2.0 - 1.0)[:, None])
    return pos.astype(np.float32), np.concatenate(feats, 1).T.astype(np.float32).copy()

def coverage_crops(st, k, rng, group=4):
    N = len(st["xyz"]); covered = np.zeros(N, dtype=bool)
    while not covered.all():
        todo = np.flatnonzero(~covered)
        centres = rng.choice(todo, size=min(group, len(todo)), replace=False)
        crops = [crop_indices(st, c, k) for c in centres]
        for idx in crops: covered[idx] = True
        yield crops

def predict_points(st, npts, use_intensity, prob_fn):
    """One label per point of a (sub-sampled) station: overlapping crops, summed probabilities - as predict_station() in notebook 04.
    prob_fn(P [g,n,3], X [g,C,n]) -> class probabilities [g, n, classes]."""
    N = len(st["xyz"]); score = np.zeros((N, C), dtype=np.float32); r = np.random.default_rng(0); n_crops = 0
    for crops in coverage_crops(st, npts, r, CFG["group"]):
        S = [make_sample(st, idx, use_intensity) for idx in crops]
        prob = prob_fn(np.stack([s[0] for s in S]), np.stack([s[1] for s in S]))
        for idx, p in zip(crops, prob): score[idx] += p
        n_crops += len(crops)
    assert (score.sum(1) > 0).all()
    return score.argmax(1), n_crops

def evaluate_run(use_intensity, npts, prob_fn, voxels):
    """Confusion matrices (rows = true class, columns = predicted class) for each evaluation mode."""
    cms, info = {}, {}
    for v in voxels:
        cms[v] = np.zeros((C, C), dtype=np.int64); cms["full<-" + v] = np.zeros((C, C), dtype=np.int64); info[v] = 0
        for s in CFG["val"]:
            st = SUB[v][s]; pred, n_crops = predict_points(st, npts, use_intensity, prob_fn); info[v] += n_crops
            cms[v] += confusion(st["label"], pred, C)
            cms["full<-" + v] += confusion(FULL[s]["label"], pred[st["inv"]], C)
        assert cms[v].sum() == N_SUB[v].sum() and cms["full<-" + v].sum() == N_FULL.sum()
    return cms, info

def readout(cm, c, top=4):
    """Where the points of class c go (recall side) and how clean the predictions of class c are (precision side)."""
    row = cm[c]; n = int(row.sum()); pred_n = int(cm[:, c].sum())
    if n == 0: return f"{CLASS_NAMES[c]:<14} no points"
    parts = ", ".join(f"{CLASS_NAMES[j]} {100 * row[j] / n:.1f}%" for j in np.argsort(-row)[:top] if row[j] > 0)
    prec = f"{100 * cm[c, c] / pred_n:.1f}%" if pred_n else "n/a"
    return f"{CLASS_NAMES[c]:<14} {n:>9,} points -> predicted as: {parts}  ||  predicted as {CLASS_NAMES[c]}: {pred_n:,} points, {prec} correct"

def report_run(name, cms, note=""):
    M = {mode: metrics_from_confusion(cm, CFG["small"]) for mode, cm in cms.items()}
    print(f"\n{'=' * 100}\n{name}  |  validation = {'+'.join(CFG['val'])}  {note}\n")
    headline(list(M.items()))
    print("\nIoU % per class (the 'points' column counts ORIGINAL points)"); per_class(list(M.items()), N_FULL)
    for mode in [m for m in cms if m.startswith("full<-")]:
        print(f"\nwhere the points go - mode {mode}")
        for c in CFG["focus"]: print("  " + readout(cms[mode], c))
    return M

# self-check of the pipeline with a fake classifier (random scores): shapes, coverage, bookkeeping
_rs = np.random.default_rng(1)
_cms, _info = evaluate_run(True, 6000, lambda P, X: _rs.random((P.shape[0], P.shape[1], C), dtype=np.float32), ["2cm"])
assert _cms["2cm"].sum() == N_SUB["2cm"].sum() and _cms["full<-2cm"].sum() == N_FULL.sum()
print("prediction pipeline self-check OK |", _info["2cm"], "crops of 6,000 points covered the 2 cm validation stations")

## Model and checkpoints

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
DEV = "cuda" if torch.cuda.is_available() else "cpu"
SMOKE = DEV != "cuda"
print("PyTorch", torch.__version__, "| device:", DEV, "|", torch.cuda.get_device_name(0) if DEV == "cuda" else "SMOKE TEST (no GPU): stand-in model, numbers are meaningless")

roots = ["/kaggle/input", os.environ.get("CKPT_DIR", "")]
if CFG.get("nb04_version"):   # works in an interactive session only; a saved run fails here unless that version is the newest one
    roots.append(__import__("kagglehub").notebook_output_download(f"{CFG['nb04']}/versions/{CFG['nb04_version']}"))
CKPTS = sorted({p for r in roots if r and os.path.isdir(r) for p in glob.glob(os.path.join(r, "**", "ckpt_best.pth"), recursive=True)})
print("checkpoints found:", CKPTS if CKPTS else "NONE")

class DummySeg(nn.Module):
    """Stand-in used only for the CPU smoke test (the real model needs CUDA extensions)."""
    def __init__(self, in_ch, n_cls):
        super().__init__(); self.net = nn.Sequential(nn.Conv1d(in_ch + 3, 32, 1), nn.ReLU(), nn.Conv1d(32, n_cls, 1))
    def forward(self, data):
        return self.net(torch.cat([data["pos"].transpose(1, 2), data["x"]], 1))

def sh(cmd, cwd=None, timeout=1500):
    p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    return p.returncode == 0, "\n".join((p.stdout + "\n" + p.stderr).strip().splitlines()[-25:])

_OP = {}
def setup_openpoints():
    """Clone + compile OpenPoints once (same steps as notebook 04)."""
    if _OP: return _OP
    src = "/kaggle/temp/code" if ON_KAGGLE else "./_code"; os.makedirs(src, exist_ok=True)
    repo = os.path.join(src, "PointNeXt")
    sh("git config --global url.https://github.com/.insteadOf git@github.com:")
    if not os.path.isdir(os.path.join(repo, "openpoints", "models")):
        ok, out = sh(f"git clone --depth 1 --recurse-submodules --shallow-submodules https://github.com/guochengqian/PointNeXt {repo}")
        if not ok: raise RuntimeError("clone failed:\n" + out)
    t0 = time.time()
    ok, out = sh(f"{sys.executable} setup.py install", cwd=os.path.join(repo, "openpoints", "cpp", "pointnet2_batch"))
    if not ok: raise RuntimeError("compiling pointnet2_batch failed:\n" + out)
    print(f"compiled pointnet2_batch in {time.time() - t0:.0f} s")
    sys.path.insert(0, repo)
    for _ in range(25):
        try:
            for k in [k for k in sys.modules if k == "openpoints" or k.startswith("openpoints.")]: del sys.modules[k]
            from openpoints.utils import EasyConfig
            from openpoints.models import build_model_from_cfg
            break
        except ModuleNotFoundError as e:
            pkg = e.name.split(".")[0]
            if pkg.startswith(("openpoints", "pointnet2")): raise
            ok, out = sh(f"{sys.executable} -m pip install -q {pkg}", timeout=600); print("pip install", pkg, ok)
            if not ok: raise RuntimeError(out)
    ok, commit = sh("git log -1 --format=%h", cwd=repo)
    _OP.update(EasyConfig=EasyConfig, build=build_model_from_cfg, repo=repo, commit=commit.strip())
    return _OP

def make_model(in_ch):
    if SMOKE: return DummySeg(in_ch, C)
    op = setup_openpoints()
    cfg = op["EasyConfig"](); cfg.load(os.path.join(op["repo"], "cfgs", "s3dis", "pointnet++.yaml"), recursive=True)
    cfg["model"]["encoder_args"]["in_channels"] = in_ch
    cfg["model"]["cls_args"]["num_classes"] = C
    return op["build"](cfg.model)

def make_prob_fn(model):
    @torch.no_grad()
    def f(P, X):
        logits = model({"pos": torch.from_numpy(P).to(DEV), "x": torch.from_numpy(X).to(DEV)})      # [g, classes, n]
        return F.softmax(logits, 1).transpose(1, 2).cpu().numpy()
    return f

# what to evaluate: every checkpoint found; on CPU a stand-in so that the code path is tested
JOBS = []
if SMOKE:
    JOBS.append(dict(name="SMOKE_dummy", path=None, cfg=dict(run_name="SMOKE_dummy", use_intensity=True, npts=24000, voxel="2cm"), best=None, epoch=None))
else:
    for p in CKPTS:
        ck = torch.load(p, map_location="cpu")
        JOBS.append(dict(name=ck["cfg"]["run_name"], path=p, cfg=ck["cfg"], best=ck.get("best"), epoch=ck.get("epoch")))
        del ck
    seen = set(); JOBS = [j for j in JOBS if not (j["name"] in seen or seen.add(j["name"]))]
if not JOBS:
    print("\n" + "!" * 100 + "\nNO CHECKPOINT ATTACHED - nothing to evaluate. Add Input -> Your Work -> Notebooks -> 04_baseline_pointnet2, then run again.\n" + "!" * 100)
for j in JOBS: print("to evaluate:", j["name"], "| trained on", j["cfg"]["voxel"], "| intensity:", j["cfg"]["use_intensity"], "| best epoch:", j["epoch"], "| stored val mIoU:", j["best"])

## Evaluation

In [ ]:
RESULTS = {}
voxels = ["2cm"] + (["1cm"] if CFG["also_1cm"] else [])
for j in JOBS:
    cfg = j["cfg"]; t0 = time.time()
    model = make_model(4 + int(cfg["use_intensity"]))
    if j["path"]:
        model.load_state_dict(torch.load(j["path"], map_location="cpu")["model"])
    model = model.to(DEV).eval()
    cms, info = evaluate_run(bool(cfg["use_intensity"]), int(cfg["npts"]), make_prob_fn(model), voxels)
    M = report_run(j["name"], cms, "| SMOKE TEST - numbers are meaningless" if SMOKE else f"| best epoch {j['epoch']} | GPU run")
    if j["best"] is not None:
        d = M[cfg["voxel"]]["mIoU"] - j["best"]
        print(f"\nreproduction check: mIoU at {cfg['voxel']} now {M[cfg['voxel']]['mIoU']:.2f}, stored in the checkpoint {j['best']:.2f}, difference {d:+.2f}"
              + ("" if abs(d) < 0.5 else "   <-- LARGER THAN 0.5: the evaluation here differs from notebook 04, investigate before using these numbers"))
    od = os.path.join(OUT, j["name"]); os.makedirs(od, exist_ok=True)
    for mode, cm in cms.items(): np.save(os.path.join(od, "confusion_" + mode.replace("<-", "_from_") + ".npy"), cm)
    RESULTS[j["name"]] = dict(smoke=SMOKE, checkpoint=j["path"], best_epoch=j["epoch"], stored_val_mIoU=j["best"], cfg=cfg,
                              crops=info, seconds=round(time.time() - t0, 1), metrics=M,
                              focus={mode: {CLASS_NAMES[c]: cms[mode][c].tolist() for c in CFG["focus"]} for mode in cms if mode.startswith("full<-")})
    print(f"\n{j['name']}: {info} crops, {time.time() - t0:.0f} s")
    del model

In [ ]:
summary = dict(eval=CFG["eval_name"], cfg=CFG, class_names=CLASS_NAMES, points_original=N_FULL.tolist(),
               points_sub={v: N_SUB[v].tolist() for v in VOXELS}, oracle=ORACLE, runs=RESULTS,
               torch=torch.__version__, gpu=torch.cuda.get_device_name(0) if DEV == "cuda" else None,
               openpoints_commit=_OP.get("commit"),
               note="rows of the 'focus' lists are confusion-matrix rows: index = predicted class ID")
with open(os.path.join(OUT, "fullres_summary.json"), "w") as fh: json.dump(summary, fh, indent=1)

print("SUMMARY on the ORIGINAL validation points" + ("   (SMOKE TEST - model numbers are meaningless; the oracle rows are real)" if SMOKE else "") + "\n")
rows = [("oracle " + v, ORACLE[v]) for v in VOXELS]
for name, r in RESULTS.items():
    rows += [(f"{name} {mode}", m) for mode, m in r["metrics"].items() if mode.startswith("full<-")]
headline(rows)
print("\nIoU % per class"); per_class([(n[-10:], m) for n, m in rows], N_FULL)
print("\ncolumns, left to right:", " | ".join(n for n, _ in rows))
print("\nsaved:", sorted(os.path.relpath(os.path.join(d, f), OUT) for d, _, fs in os.walk(OUT) for f in fs))

## After the run

Copy into `docs/experiment_log.md` as **EVAL-001** (one entry per checkpoint evaluated): the headline table, the per-class table for the small components,
the "where the points go" lines, the oracle rows, and the Kaggle version number of this notebook and of the notebook-04 version whose checkpoint was attached.

How to read it:
- **oracle** far below 100 for a class → the voxel size destroys that class before the network sees it; a finer resolution (or resolution-aware sampling) is needed.
- oracle high but **full<-2cm** IoU near 0 → the network is the problem; the "where the points go" line says which class absorbs the points.
- **full<-1cm** is exploratory: the network was trained on 2 cm crops, so 1 cm crops show it a smaller neighbourhood than it was trained on.